In [ ]:
import numpy as np
import pandas as pd

from sklearn.linear_model import Ridge
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_log_error
from statsmodels.tsa.deterministic import CalendarFourier, DeterministicProcess, Fourier

import warnings
warnings.filterwarnings("ignore")

In [ ]:
path = '../input/store-sales-time-series-forecasting/'

In [ ]:
df_train = pd.read_csv(path + 'train.csv',
                       usecols=['store_nbr', 'family', 'date', 'sales'],
                       dtype={'store_nbr': 'category', 'family': 'category', 'sales': 'float32'},
                       parse_dates=['date'], infer_datetime_format=True)




In [ ]:
test = pd.read_csv(path + 'test.csv', parse_dates=["date"])

In [ ]:
oil = pd.read_csv(path + "oil.csv", parse_dates=["date"]).rename(columns={"dcoilwtico": "oil"})

In [ ]:
def cprint(title, *args):
    print(
        "="*len(title), title, "="*len(title),
        *args,
        sep="\n",
    )

In [ ]:
train_start = df_train.date.min().date()
train_end = df_train.date.max().date()
test_start = test.date.min().date()
test_end = test.date.max().date()

In [ ]:
# اطمینان از اینکه 'date' به نوع datetime تبدیل شده است
df_train['date'] = pd.to_datetime(df_train['date'])

# ایجاد دامنه کامل تاریخ‌ها
date_range = pd.date_range(start=df_train['date'].min(), end=df_train['date'].max(), freq='D')

# ایجاد DataFrame کامل با ترکیب تمام `store_nbr` و `family`
full_grid = pd.DataFrame(
    [(store, family, date) for store in df_train['store_nbr'].unique()
     for family in df_train['family'].unique()
     for date in date_range],
    columns=['store_nbr', 'family', 'date']
)

# ترکیب با DataFrame اصلی و پر کردن مقادیر خالی با 0
df_complete = full_grid.merge(df_train, on=['store_nbr', 'family', 'date'], how='left').fillna(0)

# نمایش نتیجه
print(df_complete.head())

In [ ]:
df_complete.date = df_complete.date.dt.to_period('D')
df_complete = df_complete.set_index(['store_nbr', 'family', 'date']).sort_index()

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
sales_ts_scaled = df_complete.copy()
sales_ts_scaled[sales_ts_scaled.columns] = scaler.fit_transform(sales_ts_scaled)

In [ ]:
y = sales_ts_scaled.unstack(['store_nbr', 'family']).loc["2017"]

In [ ]:
missing_oil_dates = pd.date_range(train_start, test_end).difference(oil.date)
num_missing_oil_dates = len(missing_oil_dates)
num_wknd_missing = (missing_oil_dates.weekday >= 5).sum()
total_num_wknd = (pd.date_range(train_start, test_end).weekday >= 5).sum()

In [ ]:
cprint(
    "Missing oil dates",
    f"Number of missing oil dates: {num_missing_oil_dates}",
    f"Number of weekends missing : {num_wknd_missing}",
    f"Total number of weekends   : {total_num_wknd}",
)

In [ ]:
# reindex oil data
oil = oil.merge(
    pd.DataFrame({"date": pd.date_range(train_start, test_end)}),
    on="date",
    how="outer",
).sort_values("date", ignore_index=True)

# fill missing values using linear interpolation
oil.oil = oil.oil.interpolate(method="linear", limit_direction="both")

In [ ]:
oil.head()

In [ ]:
fourier = CalendarFourier(freq='M', order=4)
dp = DeterministicProcess(
    index=y.index,
    constant=True,
    order=2,
    seasonal=True,
    additional_terms=[fourier],
    drop=True,
)
X = dp.in_sample()
X['NewYear'] = (X.index.dayofyear == 1)

In [ ]:
holiday = pd.read_csv(
    path + 'holidays_events.csv',
    usecols=['date', 'type', 'locale', 'locale_name', 'description', 'transferred'],
    
    parse_dates=['date'],
    infer_datetime_format=True
)

In [ ]:
holiday.head()

In [ ]:
national_locale_name = sorted(holiday[holiday.locale.eq("National")].locale_name.unique().tolist())
regional_locale_name = sorted(holiday[holiday.locale.eq("Regional")].locale_name.unique().tolist())
local_locale_name = sorted(holiday[holiday.locale.eq("Local")].locale_name.unique().tolist())

In [ ]:
holiday[(holiday.description.str.contains("Navidad"))&(holiday.date.dt.year.eq(2012))]

In [ ]:
store = pd.read_csv(path+"stores.csv")


In [ ]:
def process_holiday(s):
    if "futbol" in s:
        return "futbol"
    to_remove = list(set(store.city.str.lower()) | set(store.state.str.lower()))
    for w in to_remove:
        s = s.replace(w, "")
    return s

In [ ]:
holiday.description = holiday.apply(
    lambda x: x.description.lower().replace(x.locale_name.lower(), ""), 
    axis=1,
).apply(
    process_holiday
).replace(
    r"[+-]\d+|\b(de|del|traslado|recupero|puente|-)\b", "", regex=True,
).replace(
    r"\s+|-", " ", regex=True,
).str.strip()

holiday.head()

In [ ]:
# remove transferred holidays
holiday = holiday[holiday.transferred.eq(False)]

#########################################
### Saturdays designated as work days ###
#########################################
work_days = holiday[holiday.type.eq("Work Day")]
work_days = work_days[["date", "type"]].rename(
    columns={"type": "work_day"}
).reset_index(drop=True)
work_days.work_day = work_days.work_day.notna().astype(int)
# remove work days after extracting above
holiday = holiday[holiday.type!="Work Day"].reset_index(drop=True)

###################################
### local holidays (city level) ###
###################################
local_holidays = holiday[holiday.locale.eq("Local")]
local_holidays = local_holidays[["date", "locale_name", "description"]].rename(
    columns={"locale_name": "city"}
).reset_index(drop=True)
local_holidays = local_holidays[~local_holidays.duplicated()]
local_holidays = pd.get_dummies(local_holidays, columns=["description"], prefix="loc")

#######################################
### regional holidays (state level) ###
#######################################
regional_holidays = holiday[holiday.locale.eq("Regional")]
regional_holidays = regional_holidays[["date", "locale_name", "description"]].rename(
    columns={"locale_name": "state", "description": "provincializacion"}
).reset_index(drop=True)
regional_holidays.provincializacion = regional_holidays.provincializacion.eq("provincializacion").astype(int)

#########################
### national holidays ###
#########################
national_holidays = holiday[holiday.locale.eq("National")]
national_holidays = national_holidays[["date", "description"]].reset_index(drop=True)
national_holidays = national_holidays[~national_holidays.duplicated()]
national_holidays = pd.get_dummies(national_holidays, columns=["description"], prefix="nat")
# different national holidays may fall on the same day
national_holidays = national_holidays.groupby("date").sum().reset_index()
# shorten name for visualization purposes later
national_holidays = national_holidays.rename(columns={"nat_primer grito independencia": "nat_primer grito"})

national_holidays.head()

In [ ]:
work_days.date = work_days.date.dt.to_period('D')
work_days = work_days.set_index(['date']).sort_index()

In [ ]:
selected_holidays = [
    "nat_terremoto", "nat_navidad", "nat_dia la madre", "nat_dia trabajo",
    "nat_primer dia ano", "nat_futbol", "nat_dia difuntos",
]
keep_national_holidays = national_holidays[["date", *selected_holidays]]

In [ ]:
keep_national_holidays.date = keep_national_holidays.date.dt.to_period('D')
keep_national_holidays = keep_national_holidays.set_index(['date']).sort_index()

In [ ]:
X

In [ ]:
data = X.merge(
    oil, on="date", how="left",
).merge(
    work_days, on="date", how="left",
).merge(
    keep_national_holidays, on="date", how="left",
).sort_values(["date"], ignore_index=True)
holiday_list = [col for col in data if col.startswith(("loc_", "nat_", "provincializacion"))]
data["no_holiday"] = data[holiday_list].sum(axis=1).eq(0).astype(int)
# fill columns with 0s to indicate absence of holidays/events
data[["work_day", *selected_holidays,"oil"]] = data[["work_day", *selected_holidays,"oil"]].fillna(0)
data["day_of_week"] = data.date.dt.dayofweek


In [ ]:

data = data.set_index(['date']).sort_index()

In [ ]:
X=data

In [ ]:
X.head()

In [ ]:
# Create class for two algorithms
class BoostedHybrid:
    def __init__(self, model_1, model_2):
        self.model_1 = model_1
        self.model_2 = model_2
        self.y_columns = None  # store column names from fit method

In [ ]:
# Add 'fit' method to the BoostedHybrid class 
def fit(self, X, y):

    self.model_1.fit(X,y)
    y_fit = pd.DataFrame(self.model_1.predict(X),index=X.index, columns=y.columns)
    y_resid = y - y_fit
    self.model_2.fit(X, y_resid)

    # Save column names for predict method
    self.y_columns = y.columns

# Add method to class
BoostedHybrid.fit = fit

In [ ]:
# Add 'predict' method to the BoostedHybrid class 
def predict(self, X):
    y_pred1 = pd.DataFrame(self.model_1.predict(X),index=X.index, columns=self.y_columns)
    y_pred2 = pd.DataFrame(self.model_2.predict(X),index=X.index, columns=self.y_columns)     
    y_pred = y_pred1 + y_pred2 
    
    return y_pred

# Add method to class
BoostedHybrid.predict = predict

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor

# مدل اول: Ridge با استانداردسازی
ridge_pipeline = make_pipeline(StandardScaler(), Ridge(alpha=0.5))

# مدل ترکیبی
model = BoostedHybrid(
    model_1=ridge_pipeline,
    model_2=  RandomForestRegressor(n_estimators=250, random_state=0)
)

model.fit(X, y)


In [ ]:
y_pred = model.predict(X)
y_pred = y_pred.clip(0.0)

In [ ]:
from sklearn.metrics import mean_squared_log_error
import numpy as np

# اطمینان از اینکه ایندکس‌ها هم‌راستا هستند
y_pred_stacked = y_pred.stack(['store_nbr', 'family']).reset_index()
y_target_stacked = y.stack(['store_nbr', 'family']).reset_index().copy()

# اتصال پیش‌بینی‌ها به مقدار واقعی
y_target_stacked['sales_pred'] = y_pred_stacked['sales'].values

# محاسبه RMSLE برای هر family
rmsle_per_family = y_target_stacked.groupby('family').apply(
    lambda a: np.sqrt(mean_squared_log_error(a['sales'], a['sales_pred']))
)

# چاپ مجموع RMSLEها (در صورت نیاز به جمع)
print("🔹 RMSLE (sum over families):", rmsle_per_family.sum())

# نمایش RMSLE هر family جداگانه
rmsle_per_family

In [ ]:


# Assuming X_test is returned correctly with a datetime index
X_test = dp.out_of_sample(steps=16)
X_test['NewYear'] = (X_test.index.dayofyear == 1)








In [ ]:
X_test

In [ ]:

X_test = X_test.reset_index().rename(columns={'index': 'date'})

In [ ]:
X_test = X_test.set_index(['date']).sort_index()

In [ ]:
data_ = X_test.merge(
    oil, on="date", how="left",
).merge(
    work_days, on="date", how="left",
).merge(
    keep_national_holidays, on="date", how="left",
).sort_values(["date"], ignore_index=True)
holiday_list = [col for col in data_ if col.startswith(("loc_", "nat_", "provincializacion"))]
data_["no_holiday"] = data_[holiday_list].sum(axis=1).eq(0).astype(int)
# fill columns with 0s to indicate absence of holidays/events
data_[["work_day", *selected_holidays,"oil"]] = data_[["work_day", *selected_holidays,"oil"]].fillna(0)
data_["day_of_week"] = data_.date.dt.dayofweek

In [ ]:
data_ = data_.set_index(['date']).sort_index()

In [ ]:
data.columns

In [ ]:
data_.columns

In [ ]:
X_test=data_

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Assuming scaler_y is the scaler used for the target variable during training
# and model is your trained model 

# Predict sales using the model
sales_pred = pd.DataFrame(model.predict(X_test), index=X_test.index, columns=y.columns)

# Reshape the DataFrame for inverse transformation if necessary
sales_pred_stacked = sales_pred.stack(['store_nbr', 'family']).reset_index()

# Perform inverse transformation
sales_pred_values = scaler.inverse_transform(sales_pred_stacked['sales'].values.reshape(-1, 1))

# Add the rescaled predictions back to the DataFrame
sales_pred_stacked['sales'] = sales_pred_values

# Load the submission format
df_sub = pd.read_csv(path + 'sample_submission.csv', index_col='id')

# Set the rescaled sales predictions in df_sub
df_sub['sales'] = sales_pred_stacked['sales'].values

# Save the final submission file
df_sub.to_csv('submission.csv', index=True)

print("The file submission.csv was successfully created and is ready for submission.")

In [ ]:
df_sub